# Exploratory Data Analytics for Microchannel Hotspot-Cooling CFD Data

**Purpose:** combine the data-understanding and statistical-analysis code from `AI_Prediction_Model.ipynb` and `descriptive_statistics.ipynb` into one sequential workflow.

This notebook covers dataset inspection, file/parameter identification, duplicate checking, spatial-data construction, descriptive statistics, distribution analysis, parameter sensitivity, correlation, and Kn–TMAC interaction analysis.

**Not included:** ANN architecture, model fitting, cross-validation, prediction generation, or model-performance evaluation. Those belong in a separate modelling/evaluation notebook.

**Input:** place the simulation CSV files in a folder named `hotspot_data` beside this notebook. The expected filename pattern is `2dgridhot...(Pe)...(kn...)(sv_...).csv`.

## 1. Import libraries and define paths

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

data_folder = Path("hotspot_data")
results_folder = Path("results/data_analytics")
results_folder.mkdir(parents=True, exist_ok=True)

if not data_folder.exists():
    raise FileNotFoundError(
        f"Data folder not found: {data_folder.resolve()}\n"
        "Create 'hotspot_data' and place the CFD CSV files inside it."
    )

csv_files = sorted(data_folder.glob("*.csv"))
print("Number of CSV files:", len(csv_files))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {data_folder.resolve()}")

for file in csv_files:
    print(file.name)

## 2. Inspect one simulation file

In [ ]:
sample_file = csv_files[0]
sample_df = pd.read_csv(sample_file, header=None)
sample_values = sample_df.to_numpy(dtype=float)

print("Sample file:", sample_file.name)
print("Matrix shape:", sample_values.shape)
print("Number of values:", sample_values.size)
print("Minimum:", np.nanmin(sample_values))
print("Maximum:", np.nanmax(sample_values))
print("Contains NaN:", np.isnan(sample_values).any())
print("Contains infinity:", np.isinf(sample_values).any())

display(sample_df.head())
display(sample_df.tail())

## 3. Extract operating parameters from filenames

In [ ]:
filename_pattern = re.compile(
    r"2dgridhot(?:study)?\((.*?)\)\(kn(.*?)\)\(sv_(.*?)\)"
)

metadata_records = []
unmatched_files = []

for file in csv_files:
    match = filename_pattern.search(file.stem)
    if match is None:
        unmatched_files.append(file.name)
        continue

    metadata_records.append({
        "Filename": file.name,
        "Pe": float(match.group(1)),
        "Kn": float(match.group(2)),
        "TMAC": float(match.group(3))
    })

metadata_df = pd.DataFrame(metadata_records)

if metadata_df.empty:
    raise ValueError("No filenames matched the expected Pe/Kn/TMAC pattern.")

metadata_df = metadata_df.sort_values(
    ["Pe", "Kn", "TMAC", "Filename"]
).reset_index(drop=True)

print("Files with identified parameters:", len(metadata_df))
print("Files not recognised:", unmatched_files)
print("Pe values:", sorted(metadata_df["Pe"].unique()))
print("Kn values:", sorted(metadata_df["Kn"].unique()))
print("TMAC values:", sorted(metadata_df["TMAC"].unique()))
display(metadata_df)

## 4. Detect duplicate parameter combinations and compare fields

In [ ]:
duplicate_groups = metadata_df[
    metadata_df.duplicated(["Pe", "Kn", "TMAC"], keep=False)
].sort_values(["Pe", "Kn", "TMAC", "Filename"])

print("Repeated parameter combinations:")
display(duplicate_groups)

# Compare the actual arrays. Files are considered duplicates only when
# their shapes and all values match exactly.
duplicate_file_names = set()
duplicate_comparisons = []

for _, group in duplicate_groups.groupby(["Pe", "Kn", "TMAC"]):
    files = group["Filename"].tolist()
    reference = pd.read_csv(data_folder / files[0], header=None).to_numpy(dtype=float)

    for other_name in files[1:]:
        other = pd.read_csv(data_folder / other_name, header=None).to_numpy(dtype=float)
        same_shape = reference.shape == other.shape
        exact_match = same_shape and np.array_equal(reference, other)
        max_difference = (
            float(np.max(np.abs(reference - other)))
            if same_shape else np.nan
        )
        duplicate_comparisons.append({
            "Reference_File": files[0],
            "Compared_File": other_name,
            "Same_Shape": same_shape,
            "Exactly_Identical": exact_match,
            "Maximum_Absolute_Difference": max_difference
        })
        if exact_match:
            duplicate_file_names.add(other_name)

duplicate_comparison_df = pd.DataFrame(duplicate_comparisons)
display(duplicate_comparison_df)

# Keep one file for each exact duplicate field. If files share parameter
# values but differ numerically, retain both and flag the ambiguity.
metadata_unique = metadata_df[
    ~metadata_df["Filename"].isin(duplicate_file_names)
].copy()

print("Files before exact-duplicate removal:", len(metadata_df))
print("Files removed as exact duplicates:", len(duplicate_file_names))
print("Files retained:", len(metadata_unique))

## 5. Build the case-level and spatial datasets

In [ ]:
# Grid dimensions used in the original CFD data
Nx = 123
Ny = 41
X_coords = np.linspace(-10, 20, Nx)
Y_coords = (np.arange(Ny) + 0.5) / Ny
XX, YY = np.meshgrid(X_coords, Y_coords)

spatial_records = []
case_arrays = []

for _, sim in metadata_unique.iterrows():
    file_path = data_folder / sim["Filename"]
    temperature = pd.read_csv(file_path, header=None).to_numpy(dtype=float)

    if temperature.shape != (Ny, Nx):
        raise ValueError(
            f"Unexpected grid shape in {sim['Filename']}: "
            f"{temperature.shape}; expected {(Ny, Nx)}"
        )
    if not np.all(np.isfinite(temperature)):
        raise ValueError(f"NaN or infinite values found in {sim['Filename']}")

    case_arrays.append({
        "Filename": sim["Filename"],
        "Pe": sim["Pe"],
        "Kn": sim["Kn"],
        "TMAC": sim["TMAC"],
        "Values": temperature.ravel()
    })

    spatial_records.append(pd.DataFrame({
        "X": XX.ravel(),
        "Y": YY.ravel(),
        "Pe": sim["Pe"],
        "Kn": sim["Kn"],
        "TMAC": sim["TMAC"],
        "Temperature": temperature.ravel(),
        "Simulation": sim["Filename"]
    }))

full_df = pd.concat(spatial_records, ignore_index=True)
cases_df = pd.DataFrame([
    {k: v for k, v in case.items() if k != "Values"}
    for case in case_arrays
])

print("Unique CFD cases:", len(cases_df))
print("Spatial observations:", len(full_df))
print("Observations per case:", Nx * Ny)
print("Spatial dataset shape:", full_df.shape)
display(full_df.head())

## 6. Dataset coverage and data-quality checks

In [ ]:
print("DATASET DIMENSIONS:", full_df.shape)
print("UNIQUE SIMULATIONS:", full_df["Simulation"].nunique())

print("\nMissing values:")
display(full_df.isna().sum())

numeric_df = full_df.select_dtypes(include=np.number)
print("\nInfinite values:")
display(pd.Series(np.isinf(numeric_df).sum(), name="Infinite_Count"))

print("\nUnique parameter values:")
for column in ["Pe", "Kn", "TMAC"]:
    print(column, sorted(full_df[column].unique()))

case_table = (
    full_df[["Pe", "Kn", "TMAC", "Simulation"]]
    .drop_duplicates()
    .sort_values(["Pe", "Kn", "TMAC"])
)
print("\nParameter combinations represented:")
display(case_table)

all_combinations = pd.MultiIndex.from_product(
    [
        sorted(full_df["Pe"].unique()),
        sorted(full_df["Kn"].unique()),
        sorted(full_df["TMAC"].unique())
    ],
    names=["Pe", "Kn", "TMAC"]
)
existing_combinations = pd.MultiIndex.from_frame(
    case_table[["Pe", "Kn", "TMAC"]].drop_duplicates()
)
missing_combinations = all_combinations.difference(existing_combinations)

print("Possible combinations in the Cartesian product:", len(all_combinations))
print("Available combinations:", len(existing_combinations))
print("Missing combinations:", len(missing_combinations))
display(missing_combinations.to_frame(index=False))

## 7. Overall temperature descriptive statistics

In [ ]:
temperature = full_df["Temperature"]
print("Temperature summary:")
display(temperature.describe())

print("Zero values:", int((temperature == 0).sum()))
print("Negative values:", int((temperature < 0).sum()))
print("Positive values:", int((temperature > 0).sum()))

quantiles = temperature.quantile(
    [0, 0.001, 0.01, 0.05, 0.10, 0.25, 0.50,
     0.75, 0.90, 0.95, 0.99, 1.0]
)
display(quantiles.rename("Temperature_Quantile").to_frame())

print("\nPercentage below selected thresholds:")
for threshold in [1e-12, 1e-10, 1e-8, 1e-6, 1e-4, 1e-2]:
    percentage = (temperature < threshold).mean() * 100
    print(f"Below {threshold:g}: {percentage:.3f}%")

## 8. Descriptive statistics for every CFD case

In [ ]:
case_statistics = []

for case in case_arrays:
    values = np.asarray(case["Values"], dtype=float)
    mean_value = np.mean(values)
    deviations = values - mean_value
    m2 = np.mean(deviations ** 2)
    m3 = np.mean(deviations ** 3)
    m4 = np.mean(deviations ** 4)

    skewness = m3 / (m2 ** 1.5) if m2 > 0 else np.nan
    excess_kurtosis = (m4 / (m2 ** 2)) - 3 if m2 > 0 else np.nan

    q1, median, q3 = np.percentile(values, [25, 50, 75])

    case_statistics.append({
        "Filename": case["Filename"],
        "Pe": case["Pe"],
        "Kn": case["Kn"],
        "TMAC": case["TMAC"],
        "N": len(values),
        "Sum": np.sum(values),
        "Mean": mean_value,
        "Median": median,
        "Minimum": np.min(values),
        "Maximum": np.max(values),
        "Range": np.max(values) - np.min(values),
        "Q1": q1,
        "Q2": median,
        "Q3": q3,
        "IQR": q3 - q1,
        "Variance": m2,
        "Std_Dev": np.sqrt(m2),
        "Skewness": skewness,
        "Excess_Kurtosis": excess_kurtosis
    })

stats_df = pd.DataFrame(case_statistics).sort_values(
    ["Pe", "Kn", "TMAC"]
).reset_index(drop=True)

print("Case-level statistics table:", stats_df.shape)
display(stats_df.round(6))

## 9. Temperature distribution plots

In [ ]:
# Overall raw-temperature histogram
plt.figure(figsize=(9, 5))
plt.hist(full_df["Temperature"], bins=100, edgecolor="black")
plt.xlabel("Dimensionless temperature")
plt.ylabel("Frequency")
plt.title("Overall temperature distribution")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Log view is used only for visualising the wide dynamic range.
positive_temperature = full_df.loc[
    full_df["Temperature"] > 0, "Temperature"
]
plt.figure(figsize=(9, 5))
plt.hist(np.log10(positive_temperature), bins=100, edgecolor="black")
plt.xlabel("log10(Dimensionless temperature)")
plt.ylabel("Frequency")
plt.title("Log-scale view of positive temperature values")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 10. Compare distributions across Pe, Kn and TMAC

In [ ]:
# Use a sample for plotting when the full spatial table is large.
plot_df = full_df.copy()

# Boxplot: Kn variation at TMAC = 1
df_kn = plot_df[np.isclose(plot_df["TMAC"], 1.0)]
plt.figure(figsize=(11, 6))
sns.boxplot(
    data=df_kn, x="Pe", y="Temperature", hue="Kn",
    showfliers=False
)
plt.title("Temperature distribution: Kn variation (TMAC = 1.0)")
plt.xlabel("Peclet number (Pe)")
plt.ylabel("Dimensionless temperature")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# Boxplot: TMAC variation at Kn = 0.1
df_tmac = plot_df[np.isclose(plot_df["Kn"], 0.1)]
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df_tmac, x="Pe", y="Temperature", hue="TMAC",
    showfliers=False
)
plt.title("Temperature distribution: TMAC variation (Kn = 0.1)")
plt.xlabel("Peclet number (Pe)")
plt.ylabel("Dimensionless temperature")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 11. Case-level parameter sensitivity: Pe, Kn and TMAC

In [ ]:
# Pe sensitivity at fixed Kn=0.1 and TMAC=1.0
pe_sensitivity = stats_df[
    np.isclose(stats_df["Kn"], 0.1) &
    np.isclose(stats_df["TMAC"], 1.0)
].sort_values("Pe")

display(pe_sensitivity[
    ["Pe", "Kn", "TMAC", "Mean", "Maximum", "Std_Dev"]
].round(6))

if not pe_sensitivity.empty:
    reference = pe_sensitivity.iloc[0]
    pe_change = pe_sensitivity[["Pe", "Mean", "Maximum", "Std_Dev"]].copy()
    for col in ["Mean", "Maximum", "Std_Dev"]:
        pe_change[f"{col}_Change_%"] = (
            (pe_change[col] - reference[col]) / reference[col] * 100
        )
    display(pe_change.round(4))

# Kn sensitivity at Pe=10, TMAC=1.0
kn_sensitivity = stats_df[
    np.isclose(stats_df["Pe"], 10) &
    np.isclose(stats_df["TMAC"], 1.0)
].sort_values("Kn")
display(kn_sensitivity[
    ["Pe", "Kn", "TMAC", "Mean", "Maximum", "Std_Dev"]
].round(6))

# TMAC sensitivity at Pe=10, Kn=0.1
tmac_sensitivity = stats_df[
    np.isclose(stats_df["Pe"], 10) &
    np.isclose(stats_df["Kn"], 0.1)
].sort_values("TMAC")
display(tmac_sensitivity[
    ["Pe", "Kn", "TMAC", "Mean", "Maximum", "Std_Dev"]
].round(6))

## 12. Percentage-change sensitivity across available Pe levels

In [ ]:
# Kn: compare Kn=0.1 against Kn=0.01 at each Pe, for TMAC=1.0
kn_data = stats_df[np.isclose(stats_df["TMAC"], 1.0)].copy()
kn_reference = kn_data[
    np.isclose(kn_data["Kn"], 0.01)
][["Pe", "Mean", "Maximum", "Std_Dev"]]

kn_comparison = kn_data.merge(
    kn_reference, on="Pe", suffixes=("", "_ref")
)
for col in ["Mean", "Maximum", "Std_Dev"]:
    kn_comparison[f"{col}_Change_%"] = (
        (kn_comparison[col] - kn_comparison[f"{col}_ref"])
        / kn_comparison[f"{col}_ref"] * 100
    )
display(kn_comparison.sort_values(["Pe", "Kn"])[
    ["Pe", "Kn", "Mean", "Mean_Change_%",
     "Maximum", "Maximum_Change_%",
     "Std_Dev", "Std_Dev_Change_%"]
].round(4))

# TMAC: compare each available TMAC against TMAC=1.0 at each Pe, for Kn=0.1
tmac_data = stats_df[np.isclose(stats_df["Kn"], 0.1)].copy()
tmac_reference = tmac_data[
    np.isclose(tmac_data["TMAC"], 1.0)
][["Pe", "Mean", "Maximum", "Std_Dev"]]

tmac_comparison = tmac_data.merge(
    tmac_reference, on="Pe", suffixes=("", "_ref")
)
for col in ["Mean", "Maximum", "Std_Dev"]:
    tmac_comparison[f"{col}_Change_%"] = (
        (tmac_comparison[col] - tmac_comparison[f"{col}_ref"])
        / tmac_comparison[f"{col}_ref"] * 100
    )
display(tmac_comparison.sort_values(["Pe", "TMAC"])[
    ["Pe", "TMAC", "Mean", "Mean_Change_%",
     "Maximum", "Maximum_Change_%",
     "Std_Dev", "Std_Dev_Change_%"]
].round(4))

## 13. Correlation and relationship analysis

In [ ]:
correlation_columns = [
    "Pe", "Kn", "TMAC", "Mean", "Median",
    "Std_Dev", "IQR", "Maximum", "Skewness", "Excess_Kurtosis"
]
corr_df = stats_df[correlation_columns].replace(
    [np.inf, -np.inf], np.nan
).dropna()

pearson_corr = corr_df.corr(method="pearson")
spearman_corr = corr_df.corr(method="spearman")

print("Pearson correlation (linear association):")
display(pearson_corr.round(3))

print("Spearman correlation (rank/monotonic association):")
display(spearman_corr.round(3))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.heatmap(
    pearson_corr, annot=True, cmap="coolwarm",
    vmin=-1, vmax=1, center=0, fmt=".2f", ax=axes[0]
)
axes[0].set_title("Pearson correlation")
sns.heatmap(
    spearman_corr, annot=True, cmap="coolwarm",
    vmin=-1, vmax=1, center=0, fmt=".2f", ax=axes[1]
)
axes[1].set_title("Spearman correlation")
plt.tight_layout()
plt.show()

## 14. Kn–TMAC interaction analysis

In [ ]:
# Plot how the Kn trend changes with TMAC for each Pe.
outputs = ["Mean", "Maximum", "Std_Dev"]
clean_stats = stats_df.drop_duplicates(
    subset=["Pe", "Kn", "TMAC"], keep="first"
).copy()

pe_values = sorted(clean_stats["Pe"].unique())
tmac_values = sorted(clean_stats["TMAC"].unique())

for output in outputs:
    fig, axes = plt.subplots(
        1, len(pe_values),
        figsize=(max(5 * len(pe_values), 10), 4),
        squeeze=False
    )
    for i, pe in enumerate(pe_values):
        ax = axes[0, i]
        subset = clean_stats[np.isclose(clean_stats["Pe"], pe)]
        for tmac in tmac_values:
            selected = subset[np.isclose(subset["TMAC"], tmac)].sort_values("Kn")
            if not selected.empty:
                ax.plot(
                    selected["Kn"], selected[output],
                    marker="o", linewidth=2, label=f"TMAC={tmac:g}"
                )
        ax.set_title(f"Pe = {pe:g}")
        ax.set_xlabel("Kn")
        ax.set_ylabel(output)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.suptitle(f"Kn–TMAC interaction: {output}")
    plt.tight_layout()
    plt.show()

## 15. Quantify Kn–TMAC interaction using matched comparisons

In [ ]:
# Compare Kn=0.01 and Kn=0.1 at each Pe and TMAC, only where both exist.
interaction_records = []

for pe in sorted(clean_stats["Pe"].unique()):
    subset_pe = clean_stats[np.isclose(clean_stats["Pe"], pe)]
    for tmac in sorted(subset_pe["TMAC"].unique()):
        selected = subset_pe[np.isclose(subset_pe["TMAC"], tmac)]
        low = selected[np.isclose(selected["Kn"], 0.01)]
        high = selected[np.isclose(selected["Kn"], 0.1)]

        if len(low) == 1 and len(high) == 1:
            for output in outputs:
                low_value = low[output].iloc[0]
                high_value = high[output].iloc[0]
                interaction_records.append({
                    "Pe": pe,
                    "TMAC": tmac,
                    "Output": output,
                    "Value_Kn_0.01": low_value,
                    "Value_Kn_0.1": high_value,
                    "Absolute_Change": high_value - low_value,
                    "Percentage_Change": (
                        (high_value - low_value) / low_value * 100
                        if low_value != 0 else np.nan
                    )
                })

interaction_df = pd.DataFrame(interaction_records)
display(interaction_df.round(6))

## 16. Export cleaned case statistics and analysis tables

In [ ]:
stats_output = results_folder / "case_level_descriptive_statistics.csv"
coverage_output = results_folder / "parameter_coverage.csv"
interaction_output = results_folder / "kn_tmac_interaction.csv"

stats_df.to_csv(stats_output, index=False)
case_table.to_csv(coverage_output, index=False)
interaction_df.to_csv(interaction_output, index=False)

excel_output = results_folder / "CFD_Data_Analytics_Summary.xlsx"
with pd.ExcelWriter(excel_output, engine="openpyxl") as writer:
    stats_df.to_excel(writer, sheet_name="Case_Statistics", index=False)
    case_table.to_excel(writer, sheet_name="Parameter_Coverage", index=False)
    duplicate_comparison_df.to_excel(writer, sheet_name="Duplicate_Check", index=False)
    pe_sensitivity.to_excel(writer, sheet_name="Pe_Sensitivity", index=False)
    kn_comparison.to_excel(writer, sheet_name="Kn_Sensitivity", index=False)
    tmac_comparison.to_excel(writer, sheet_name="TMAC_Sensitivity", index=False)
    interaction_df.to_excel(writer, sheet_name="Kn_TMAC_Interaction", index=False)
    pearson_corr.to_excel(writer, sheet_name="Pearson_Correlation")
    spearman_corr.to_excel(writer, sheet_name="Spearman_Correlation")

print("Saved:")
print(stats_output)
print(coverage_output)
print(interaction_output)
print(excel_output)

## Notes on interpretation

- A spatial row is one grid point, not an independent CFD simulation. The independent case count is the number of unique simulation files/cases.
- The parameter combinations are not necessarily a complete factorial design. Interpret sensitivity only for combinations that are present.
- Correlation describes association; it does not establish causality.
- The duplicate-removal step removes a file only when its temperature field is exactly identical to another file with the same Pe, Kn and TMAC.
- The analysis uses the grid dimensions and coordinate definitions present in the original project notebook (`123 × 41`, X from -10 to 20, cell-centred Y from 0 to 1).
- Model training, cross-validation, prediction-error analysis and ANN comparisons are deliberately kept out of this EDA notebook.